## Data generation

In [2]:
import numpy as np

# Function that generates the BMT
def create_brownian_with_a_trend(
    n_paths, n_points, mu, sigma, seed
):
    dt = 1/n_points
    X = np.zeros((n_paths,n_points))
    np.random.seed(seed)
    for i in range(n_paths):
        dW = np.random.normal(
            0,
            np.sqrt(dt),
            n_points - 1
        )
        X[i, 1:] = np.cumsum(mu * dt + sigma * dW)
    return X

In [ ]:
import matplotlib.pyplot as plt

# First dataset
X_1 = create_brownian_with_a_trend(
    n_paths=100, n_points=20, mu=1, sigma=0.2, seed=42
)

plt.figure(figsize=(12, 6))
plt.plot(X_1.T, marker='o')
plt.xticks(range(X_1.shape[1]))
plt.savefig("bmt_1.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Second dataset
X_2 = create_brownian_with_a_trend(
    n_paths=100, n_points=15, mu=-1, sigma=0.2, seed=38
)
plt.figure(figsize=(12, 6))
plt.plot(X_2.T, marker='o')
plt.xticks(range(X_2.shape[1]))
plt.savefig("bmt_2.pdf", bbox_inches="tight")
plt.show()

## Load the data

In [ ]:
from scentree.io.loader import Dataset, DatasetsLoader

datasets = [
    Dataset(
        name="first_dataset",
        values=X_1,
        stage_ids=6*[1] + 5*[2] + 9*[3]
    ),
    Dataset(
        name="second_dataset",
        values=X_2,
        stage_ids=10*[2] + 5*[3]
    )
]
dsl = DatasetsLoader(datasets=datasets)
full_values = dsl.get_full_values()
num_vars_per_stage = dsl.get_num_variables_per_stage()
stage_cols_map = dsl.create_stage_columns_mapping()
stage_ids = dsl.get_sorted_stage_ids()
map_columns_names = dsl.create_dataset_column_mapping()

## Scenario Fan

In [ ]:
from scentree.fan_generator import StageManager

num_fans = 1
num_scenarios = 3
stage_manager = StageManager()
scenario_fans = stage_manager.generate_scenario_fans(
    X=full_values,
    num_fans=num_fans,
    num_scenarios=num_scenarios,
    build_in_sample_fans=True,
    seed=123
)

In [ ]:
cols_1 = map_columns_names[0]["columns"]
fan_1 = scenario_fans["scenarios"][0][:, cols_1]
plt.figure(figsize=(12, 6))
plt.plot(fan_1.T, linestyle='-', marker='o')
plt.plot(
    X_1[-1, :],
    linestyle='--',
    color='black',
    linewidth=2,
    marker='x'
)
plt.xticks(range(X_1.shape[1]))
plt.savefig("fan_1.pdf", bbox_inches="tight")
plt.show()

In [ ]:
cols_2 = map_columns_names[1]["columns"]
fan_2 = scenario_fans["scenarios"][0][:, cols_2]
plt.figure(figsize=(12, 6))
plt.plot(fan_2.T, linestyle='-', marker='o')
plt.plot(
    X_2[-1, :],
    linestyle='--',
    color='black',
    linewidth=2,
    marker='x'
)
plt.xticks(range(X_2.shape[1]))
plt.savefig("fan_2.pdf", bbox_inches="tight")
plt.show()

## Scenario tree

In [ ]:
from scentree.tree_construction.ftc import FTC

tree = FTC(
    scenarios=scenario_fans["scenarios"],
    num_variables_per_stage=num_vars_per_stage,
    stage_ids=stage_ids
)
tree_structure = tree.generate_scenario_trees(
    r=1.,
    stage_columns_mapping=stage_cols_map,
    initial_stage_id_to_cluster=1,
    seed=15,
)